# A.3 文件多了怎麼找相關段落？


查詢「書店地址」，兩份文件是d1「貓喜歡曬太陽」和d2「書店地址是青街8號」。最簡單的找法是看共同片段：書、店、地、址在d2出現，四項都吻合；d1一項也沒有。這個分數能排序，還不是答案正確的機率。

`lexical_terms` 把英文轉小寫，中文按單字，英文與數字按連續片段。每項貢獻取查詢和文件次數較小者，再加總；查詢的書只一次，文件重複十次也只得1。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.retrieval import lexical_terms, retrieve

query = "書店地址"
docs = [
    {"id": "d1", "text": "貓喜歡曬太陽"},
    {"id": "d2", "text": "書店地址是青街8號"},
]
print("查詢片段", lexical_terms(query))
hits = retrieve(query, docs, k=1)
print("取回", hits)
print("改問法", retrieve("營業處在哪", docs, k=1))

`k=1`最多取一份，應取d2。改問「營業處在哪」則沒有共同片段，回空清單；同義意思沒有被這份字詞規則學到。零分文件不回傳，同分按id排序。

把d2改成「書店地址尚未公佈」，分數仍4，卻不能回答地址。這一個反例就分清了「被選中」與「資料足夠」。更好的排序可以使用詞權重或語意特徵，但仍需拿有指定來源的題目核對命中，不能從有非空結果就跳到可靠答案。

<details>
<summary>補充：舊公告庫的來源命中</summary>

上述正式組把12份新公告混入60份其他店公告，共72份，這個字詞檢索器一次取一份。12題中九題用`K017`這類原店名查詢，三題刻意換成`shop003`這類它不懂的別名；正確來源命中9/12。這三次miss是已知的查詢限制，不是語言模型生成錯誤。檢索文件的持久編號如`doc-K017`，與放進提示的短引用編號如`D1`的對應也逐題保存，避免改了標籤卻無法回查。

</details>
